---

## 📦 Imports et Configuration

In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Deepchecks NLP
from deepchecks.nlp import TextData
from deepchecks.nlp.suites import data_integrity

# ML
from sklearn.model_selection import train_test_split

In [4]:
# Configuration des chemins
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'testing' else Path.cwd()
PROCESSOR_DIR = BASE_DIR / 'processors'
TESTING_DIR = BASE_DIR / 'testing'
TESTING_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("📊 DEEPCHECKS NLP - NIVEAU 1 : INTÉGRITÉ DES DONNÉES")
print("="*80)
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 Base: {BASE_DIR}")
print()

📊 DEEPCHECKS NLP - NIVEAU 1 : INTÉGRITÉ DES DONNÉES
📅 Date: 2025-12-15 01:39:36
📁 Base: e:\MLOps\mlops_election



---

## 📥 Chargement des Données

In [5]:
def load_cleaned_texts():
    """Charge les textes nettoyés"""
    texts_path = PROCESSOR_DIR / 'cleaned_texts.pkl'
    if not texts_path.exists():
        raise FileNotFoundError(f"Textes non trouvés: {texts_path}")
    
    with open(texts_path, 'rb') as f:
        data = pickle.load(f)
    
    print(f"✅ Textes chargés: {len(data['cleaned'])} textes")
    return data['cleaned'], data['labels']

In [6]:
# Charger les données
texts, labels = load_cleaned_texts()

✅ Textes chargés: 3434 textes


---

## 📝 Création des TextData pour Deepchecks NLP

In [7]:
def create_text_data(texts_list, labels_list, split_name='train'):
    """Crée un TextData Deepchecks NLP à partir de textes et labels"""
    print(f"📝 Création TextData NLP ({split_name})")
    print("-" * 80)
    
    text_data = TextData(
        raw_text=texts_list,
        label=labels_list,
        task_type='text_classification',
        name=f'{split_name}_dataset'
    )
    
    print(f"✅ TextData créé:")
    print(f"   Nombre de textes: {len(texts_list)}")
    print(f"   Distribution labels: {pd.Series(labels_list).value_counts().to_dict()}")
    print()
    
    return text_data

In [8]:
# Créer le split train/test (même split que preprocess.py)
df = pd.DataFrame({'texts': texts, 'labels': labels})
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['labels']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['labels']
)

print(f"Split effectué:")
print(f"  Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
print()

Split effectué:
  Train: 2403 (70.0%)
  Val:   515 (15.0%)
  Test:  516 (15.0%)



In [9]:
# Créer les TextData NLP
train_text_data = create_text_data(
    train_df['texts'].tolist(), 
    train_df['labels'].tolist(), 
    'train'
)
test_text_data = create_text_data(
    test_df['texts'].tolist(), 
    test_df['labels'].tolist(), 
    'test'
)

📝 Création TextData NLP (train)
--------------------------------------------------------------------------------
✅ TextData créé:
   Nombre de textes: 2403
   Distribution labels: {0: 1233, 1: 1170}

📝 Création TextData NLP (test)
--------------------------------------------------------------------------------
✅ TextData créé:
   Nombre de textes: 516
   Distribution labels: {0: 265, 1: 251}



---

## 📊 NIVEAU 1 : TEXT DATA INTEGRITY

### Suite d'intégrité Deepchecks NLP

La suite `data_integrity()` exécute automatiquement les checks suivants :
1. **Text Property Outliers** : Détecte les textes avec des propriétés anormales
2. **Unknown Tokens** : Identifie les tokens jamais vus dans le vocabulaire
3. **Text Duplicates** : Trouve les textes dupliqués dans le dataset
4. **Conflicting Labels** : Détecte les textes identiques avec labels différents
5. **Property Label Correlation** : Vérifie la corrélation propriétés/labels

In [10]:
def run_text_integrity_checks(train_data, test_data):
    """NIVEAU 1: Vérifications d'intégrité des données textuelles (NLP natif)"""
    print("\n" + "="*80)
    print("📊 NIVEAU 1: TEXT DATA INTEGRITY (NLP Natif)")
    print("="*80)
    
    # Suite d'intégrité NLP
    integrity_suite = data_integrity()
    
    print("\n🔍 Checks NLP exécutés:")
    print("   1. Text Property Outliers (longueur, mots rares, etc.)")
    print("   2. Unknown Tokens (tokens jamais vus)")
    print("   3. Text Duplicates (textes dupliqués)")
    print("   4. Conflicting Labels (même texte, labels différents)")
    print("   5. Property Label Correlation")
    
    # Exécuter la suite
    print("\n⏳ Exécution des checks d'intégrité NLP...")
    result = integrity_suite.run(train_data, test_data)
    
    # Sauvegarder le rapport
    integrity_report_path = TESTING_DIR / 'deepchecks_nlp_integrity_report.html'
    result.save_as_html(str(integrity_report_path))
    
    print(f"✅ Rapport d'intégrité NLP sauvegardé: {integrity_report_path.name}")
    
    # Résumé des résultats
    print("\n📈 Résumé Intégrité NLP:")
    passed = 0
    total = 0
    for check_result in result.results:
        if hasattr(check_result, 'passed_conditions'):
            total += 1
            if check_result.passed_conditions():
                passed += 1
    
    if total > 0:
        print(f"   Checks réussies: {passed}/{total}")
    else:
        print(f"   Checks exécutés: {len(result.results)}")
    
    # Statistiques texte
    train_texts = train_data.text
    test_texts = test_data.text
    
    print("\n📝 Statistiques Texte:")
    print(f"   Train - Longueur moyenne: {np.mean([len(t) for t in train_texts]):.1f} caractères")
    print(f"   Test  - Longueur moyenne: {np.mean([len(t) for t in test_texts]):.1f} caractères")
    print(f"   Train - Mots moyens: {np.mean([len(t.split()) for t in train_texts]):.1f}")
    print(f"   Test  - Mots moyens: {np.mean([len(t.split()) for t in test_texts]):.1f}")
    
    return result

In [11]:
# Exécuter les checks d'intégrité NLP
integrity_result = run_text_integrity_checks(train_text_data, test_text_data)


📊 NIVEAU 1: TEXT DATA INTEGRITY (NLP Natif)



🔍 Checks NLP exécutés:
   1. Text Property Outliers (longueur, mots rares, etc.)
   2. Unknown Tokens (tokens jamais vus)
   3. Text Duplicates (textes dupliqués)
   4. Conflicting Labels (même texte, labels différents)
   5. Property Label Correlation

⏳ Exécution des checks d'intégrité NLP...


✅ Rapport d'intégrité NLP sauvegardé: deepchecks_nlp_integrity_report.html

📈 Résumé Intégrité NLP:
   Checks réussies: 9/10

📝 Statistiques Texte:
   Train - Longueur moyenne: 82.3 caractères
   Test  - Longueur moyenne: 83.1 caractères
   Train - Mots moyens: 15.3
   Test  - Mots moyens: 15.4


In [12]:
# Afficher le widget interactif
integrity_result

Accordion(children=(VBox(children=(HTML(value='\n<h1 id="summary_EFZ8HP0T62T4RPJTD4CHEXPSZ">Data Integrity Sui…

---

## 📊 Résumé et Recommandations

In [13]:
print("\n" + "="*80)
print("✅ NIVEAU 1 : INTÉGRITÉ DES DONNÉES - TERMINÉ")
print("="*80)

print("\n📂 Rapport généré:")
print(f"   {TESTING_DIR / 'deepchecks_nlp_integrity_report.html'}")

print("\n💡 Interprétation des résultats:")
print("   ✅ Tous les checks réussis : Données de bonne qualité")
print("   ⚠️  Text Duplicates : Vérifier si nécessaire de dédupliquer")
print("   ⚠️  Conflicting Labels : Nettoyer les annotations incohérentes")
print("   ⚠️  Property Outliers : Analyser les textes anormaux")

print("\n🔗 Ouvrez le rapport HTML pour visualiser les détails")
print("="*80)


✅ NIVEAU 1 : INTÉGRITÉ DES DONNÉES - TERMINÉ

📂 Rapport généré:
   e:\MLOps\mlops_election\testing\deepchecks_nlp_integrity_report.html

💡 Interprétation des résultats:
   ✅ Tous les checks réussis : Données de bonne qualité
   ⚠️  Text Duplicates : Vérifier si nécessaire de dédupliquer
   ⚠️  Conflicting Labels : Nettoyer les annotations incohérentes
   ⚠️  Property Outliers : Analyser les textes anormaux

🔗 Ouvrez le rapport HTML pour visualiser les détails


---

## 📚 Documentation

### Checks exécutés

| Check | Description | Critère |
|-------|-------------|---------|
| **Text Property Outliers** | Détecte les textes avec propriétés anormales | Pas de valeurs extrêmes |
| **Unknown Tokens** | Tokens jamais vus | < 5% inconnus |
| **Text Duplicates** | Textes dupliqués | < 1% duplications |
| **Conflicting Labels** | Même texte, labels différents | Aucun conflit |
| **Property Label Correlation** | Corrélation propriétés/labels | Équilibrée |

### Prochaines étapes

➡️ **NIVEAU 2** : Exécuter `deepchecks_distribution.ipynb` pour détecter le drift  
➡️ **NIVEAU 3** : Exécuter `deepchecks_performance.ipynb` pour évaluer le modèle

### Ressources
- [Deepchecks NLP Docs](https://docs.deepchecks.com/stable/nlp/index.html)
- [Data Integrity Suite](https://docs.deepchecks.com/stable/nlp/auto_checks/data_integrity/index.html)